# Day 10: Capstone - The NovaTech AI Assistant

**LangChain Hands-on Series | Day 10 of 10**
Trainer: Ajeetkumar

---

You made it to the last day. Today there are no new concepts. Instead, we build one complete application, step by step, using what you learned over the last nine days. This is the kind of project you can put on your GitHub and explain in an interview.

### The brief

NovaTech wants one assistant on its intranet that can:

1. answer questions about company **policies** (leave, IT security, travel, code of conduct) and the **CloudDesk product**, with sources
2. tell an employee **their own** leave balance
3. look up **customer orders** for the support team
4. do quick **calculations** (leave left, refunds)
5. **create support tickets** - but only after a human approves
6. remember the conversation, protect personal data, and keep costs visible

### Which day each piece comes from

| Feature | Day |
|---------|-----|
| Chat model, messages | 1 |
| Prompts, structured output | 2 |
| Chains and Runnables | 3 |
| Memory (checkpointer, thread_id) | 4, 8 |
| Loading, splitting, embeddings, vector store | 5 |
| RAG with sources | 6 |
| Tools | 7 |
| Agent, context, middleware, human approval | 8 |
| Cost tracking, guardrails, monitoring | 9 |

### Architecture

```
                                   +------------------------------------------------+
                                   |               NovaTech Assistant               |
   employee / support agent        |                                                |
   ----------------------------->  |  Middleware: PII redaction -> summarisation -> |
   question + who they are         |              call limit -> tool logging        |
   (context: employee_id, name)    |                                                |
                                   |                 AGENT (LLM)                    |
                                   |      decides which tool(s) to use              |
                                   +-------+--------+--------+--------+-------------+
                                           |        |        |        |
                              search_knowledge  my_leave  lookup   calculator   create_ticket
                                   (RAG)        _balance   _order                (needs approval)
                                     |             |         |
                                Chroma DB     employees   orders
                               (policies)       .csv       .csv
```

Let's build it.

## Step 1: Configuration

All settings in one place, so changing the model or chunk size later is a one-line change.

In [ ]:
import os
import re
import json
import warnings
from dataclasses import dataclass
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
load_dotenv()

CONFIG = {
    "model": os.getenv("OPENAI_MODEL", "gpt-4.1-nano"),
    "embedding_model": "text-embedding-3-small",
    "chunk_size": 500,
    "chunk_overlap": 60,
    "top_k": 3,
    "vector_db_dir": "chroma_capstone",
}
CONFIG

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

llm = ChatOpenAI(model=CONFIG["model"], temperature=0, timeout=30, max_retries=2)
embeddings = OpenAIEmbeddings(model=CONFIG["embedding_model"])

## Step 2: The knowledge base (Days 5 and 6)

We load every policy file plus the product FAQ, split them, and store them in Chroma on disk. Using fixed ids means that re-running this cell updates the store instead of creating duplicates.

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

def load_documents():
    docs = DirectoryLoader("data/policies", glob="*.txt", loader_cls=TextLoader,
                           loader_kwargs={"encoding": "utf-8"}).load()
    docs += TextLoader("data/policies/product_faq.md", encoding="utf-8").load()
    docs += PyPDFLoader("data/policies/code_of_conduct.pdf").load()
    for d in docs:
        d.metadata["file"] = os.path.basename(d.metadata["source"])
    return docs

documents = load_documents()
chunks = RecursiveCharacterTextSplitter(
    chunk_size=CONFIG["chunk_size"], chunk_overlap=CONFIG["chunk_overlap"]
).split_documents(documents)

knowledge_base = Chroma.from_documents(
    chunks, embeddings,
    collection_name="novatech_capstone",
    persist_directory=CONFIG["vector_db_dir"],
    ids=[f"kb-{i}" for i in range(len(chunks))],
)
retriever = knowledge_base.as_retriever(search_kwargs={"k": CONFIG["top_k"]})

print(f"{len(documents)} documents -> {len(chunks)} chunks stored in Chroma")

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

per_file = Counter(c.metadata["file"] for c in chunks)

plt.figure(figsize=(6, 3))
plt.barh(list(per_file.keys()), list(per_file.values()), color="#4C72B0")
plt.xlabel("Chunks")
plt.title("What's inside the knowledge base")
plt.show()

## Step 3: The tools (Days 7 and 8)

Each tool does one clear job and has a docstring that says *when* to use it. Notice `my_leave_balance` reads the employee id from the **context**, not from the model - so nobody can ask for someone else's balance.

In [ ]:
import pandas as pd
from langchain.tools import tool, ToolRuntime

employees = pd.read_csv("data/employees.csv")
orders = pd.read_csv("data/orders.csv")

@dataclass
class UserContext:
    employee_id: str
    name: str

In [ ]:
@tool
def search_knowledge_base(query: str) -> str:
    """Search NovaTech policies (leave, IT security, travel, code of conduct) and the CloudDesk product FAQ.
    Use this for ANY question about company rules or the CloudDesk product."""
    found = retriever.invoke(query)
    if not found:
        return "No relevant documents found."
    return "\n\n".join(f"[source: {d.metadata['file']}]\n{d.page_content}" for d in found)

The leave tool reads the employee id from the runtime context:

In [ ]:
@tool
def my_leave_balance(runtime: ToolRuntime[UserContext]) -> str:
    """Get the annual and sick leave balance of the employee who is asking."""
    row = employees[employees["employee_id"] == runtime.context.employee_id]
    if row.empty:
        return "Employee record not found."
    r = row.iloc[0]
    return f"Annual leave left: {r['annual_leave_balance']} days. Sick leave left: {r['sick_leave_balance']} days."

Order lookup and a safe calculator, the same pattern as Day 7:

In [ ]:
@tool
def lookup_order(order_id: str) -> str:
    """Look up a CloudDesk customer order by id (format ORD-1234): customer, plan, status, amount, date."""
    row = orders[orders["order_id"].str.upper() == order_id.strip().upper()]
    return row.iloc[0].to_json() if not row.empty else f"No order found with id {order_id}."

@tool
def calculator(expression: str) -> str:
    """Evaluate a maths expression such as '24 - 5' or '2520 * 340 / 365'."""
    if not re.fullmatch(r"[0-9+\-*/(). %]+", expression):
        return "Error: only numbers and + - * / ( ) % are allowed."
    return str(round(eval(expression), 2))

The ticket tool. It is the only tool that *changes* something, which is why it gets human approval in Step 4.

In [ ]:
TICKETS = []

@tool
def create_support_ticket(title: str, description: str, priority: str) -> str:
    """Create a support ticket for IT or the CloudDesk support team. priority must be low, medium or high.
    Only use this when the user asks for a ticket or reports a problem that needs follow-up."""
    ticket_id = f"TCK-{len(TICKETS) + 1:04d}"
    TICKETS.append({"id": ticket_id, "title": title, "priority": priority, "description": description})
    return f"Ticket {ticket_id} created with priority {priority}."

TOOLS = [search_knowledge_base, my_leave_balance, lookup_order, calculator, create_support_ticket]

Always test tools on their own before giving them to the agent:

In [ ]:
print(search_knowledge_base.invoke({"query": "lost laptop"})[:250], "\n")
print(lookup_order.invoke({"order_id": "ORD-1005"}))
print(calculator.invoke({"expression": "24 - 5"}))

## Step 4: Middleware (Days 8 and 9)

| Middleware | Why |
|------------|-----|
| `dynamic_prompt` | greets the user by name and sets the rules |
| `PIIMiddleware` | emails typed by users are redacted before reaching the model |
| `SummarizationMiddleware` | long chats stay small and cheap |
| `ModelCallLimitMiddleware` | a confused agent cannot loop forever |
| `HumanInTheLoopMiddleware` | `create_support_ticket` needs approval |
| our own `wrap_tool_call` | logs every tool call for the dashboard |

In [ ]:
import time
from langchain.agents.middleware import (
    dynamic_prompt, ModelRequest, wrap_tool_call, PIIMiddleware, SummarizationMiddleware,
    ModelCallLimitMiddleware, HumanInTheLoopMiddleware,
)

@dynamic_prompt
def system_prompt(request: ModelRequest) -> str:
    user = request.runtime.context
    return (
        f"You are the NovaTech AI Assistant, helping {user.name} (employee {user.employee_id}).\n"
        "Rules:\n"
        "- For policy or product questions, ALWAYS use search_knowledge_base and name the source file.\n"
        "- Use my_leave_balance for the user's own leave; never guess numbers.\n"
        "- Use calculator for any arithmetic.\n"
        "- If the tools do not contain the answer, say so and suggest contacting hr@novatech.example.\n"
        "- Be friendly and concise (max 80 words)."
    )

TOOL_LOG = []                # tool calls of the current question (cleared during evaluation)
ALL_TOOL_CALLS = Counter()   # every tool call in this session, for the dashboard

@wrap_tool_call
def log_tools(request, handler):
    start = time.time()
    result = handler(request)
    TOOL_LOG.append({"tool": request.tool_call["name"], "seconds": round(time.time() - start, 3)})
    ALL_TOOL_CALLS[request.tool_call["name"]] += 1
    print(f"   [tool] {request.tool_call['name']}({request.tool_call['args']})")
    return result

MIDDLEWARE = [
    system_prompt,
    PIIMiddleware("email", strategy="redact", apply_to_input=True),
    SummarizationMiddleware(model=llm, trigger=("tokens", 3000), keep=("messages", 10)),
    ModelCallLimitMiddleware(run_limit=8),
    HumanInTheLoopMiddleware(interrupt_on={"create_support_ticket": True}),
    log_tools,
]

## Step 5: Build the agent

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

assistant = create_agent(
    model=llm,
    tools=TOOLS,
    middleware=MIDDLEWARE,
    context_schema=UserContext,
    checkpointer=InMemorySaver(),
)

print(assistant.get_graph().draw_ascii())

## Step 6: A chat helper that handles approvals and tracks cost

The helper below sends a message, and if the agent pauses for approval (because it wants to create a ticket), it shows the request and resumes with the decision. In a real web app, the "approve" would be a button for a team lead. Here, `auto_approve` decides.

In [ ]:
from langgraph.types import Command
from langchain_core.callbacks import get_usage_metadata_callback

USAGE_LOG = []

def chat(message, user, thread_id, auto_approve=True, return_answer=False):
    config = {"configurable": {"thread_id": thread_id}}
    print(f"\n{user.name}: {message}")

    with get_usage_metadata_callback() as cb:
        result = assistant.invoke({"messages": [{"role": "user", "content": message}]}, config, context=user)

        while "__interrupt__" in result:                                 # the agent is waiting for approval
            action = result["__interrupt__"][0].value["action_requests"][0]
            print(f"   [approval needed] {action['name']} {action['args']}")
            decision = {"type": "approve"} if auto_approve else {"type": "reject", "message": "Not approved by team lead."}
            print(f"   [decision] {decision['type']}")
            result = assistant.invoke(Command(resume={"decisions": [decision]}), config, context=user)

    tokens = sum(u["total_tokens"] for u in cb.usage_metadata.values())
    USAGE_LOG.append({"message": message[:30], "tokens": tokens})
    answer = result["messages"][-1].content
    print(f"Assistant: {answer}")
    if return_answer:                     # only needed by the evaluation code
        return answer

## Step 7: Try it - real scenarios

### Scenario A: an employee planning leave

Asha (E101) asks about her balance, does some maths, and then asks a policy follow-up - all in one thread, so the assistant remembers the context.

In [ ]:
asha = UserContext(employee_id="E101", name="Asha")

chat("How many annual leave days do I have left?", asha, "asha-1")
chat("If I take 6 days in December, how many will I have left?", asha, "asha-1")
chat("How early do I need to apply for those 6 days?", asha, "asha-1")

### Scenario B: a support agent handling a customer

In [ ]:
neha = UserContext(employee_id="E105", name="Neha")

chat("What's the status of order ORD-1005?", neha, "neha-1")
chat("The customer is asking about the refund policy. What should I tell them?", neha, "neha-1")
chat("Please create a high priority ticket to follow up on this refund.", neha, "neha-1")

### Scenario C: security and safety

A lost laptop (needs a policy answer **and** a ticket), a message containing an email address (should be redacted), and a question the documents cannot answer (should not be made up).

In [ ]:
karan = UserContext(employee_id="E104", name="Karan")

chat("I lost my laptop at the airport. What should I do? Please also raise a ticket.", karan, "karan-1")
chat("Send the details to my personal email karan.shah@gmail.com", karan, "karan-1")
chat("What is the CEO's bonus this year?", karan, "karan-1")

### Scenario D: a ticket the team lead rejects

In [ ]:
chat("Create a low priority ticket: I want a second monitor.", karan, "karan-2", auto_approve=False)

In [ ]:
pd.DataFrame(TICKETS) if TICKETS else "No tickets yet"

## Step 8: Evaluate it (Day 6 style)

Before calling the project finished, we test it properly. For each test question we know **which tool should be used** and **what the answer must contain**. We check both automatically.

In [ ]:
from pydantic import BaseModel, Field

EVAL_SET = [
    {"q": "How many sick leave days do employees get per year?", "tool": "search_knowledge_base", "expect": "12 days"},
    {"q": "What is the hotel limit in metro cities?", "tool": "search_knowledge_base", "expect": "INR 6,000 per night"},
    {"q": "How many annual leave days do I have?", "tool": "my_leave_balance", "expect": "14 days"},
    {"q": "Who is the customer on order ORD-1003?", "tool": "lookup_order", "expect": "Priya Nair"},
    {"q": "What is 2520 * 340 / 365?", "tool": "calculator", "expect": "about 2347"},
    {"q": "How much is the Enterprise plan?", "tool": "search_knowledge_base", "expect": "USD 70 per agent per month"},
]

class Grade(BaseModel):
    correct: bool = Field(description="True if the answer contains the expected key fact")

judge = llm.with_structured_output(Grade)

In [ ]:
eval_user = UserContext(employee_id="E101", name="Asha")
rows = []

for i, case in enumerate(EVAL_SET):
    TOOL_LOG.clear()
    answer = chat(case["q"], eval_user, thread_id=f"eval-{i}", return_answer=True)
    used = [t["tool"] for t in TOOL_LOG]
    grade = judge.invoke(f"Question: {case['q']}\nExpected: {case['expect']}\nAnswer: {answer}\n"
                         "Does the answer contain the expected key fact?")
    rows.append({"question": case["q"][:40], "expected_tool": case["tool"],
                 "right_tool": case["tool"] in used, "correct_answer": grade.correct})

eval_df = pd.DataFrame(rows)
eval_df

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
scores = [eval_df["right_tool"].mean() * 100, eval_df["correct_answer"].mean() * 100]
ax.bar(["Right tool chosen", "Correct answer"], scores, color=["#4C72B0", "#55A868"])
ax.set_ylim(0, 100)
ax.set_ylabel("% of test cases")
for i, v in enumerate(scores):
    ax.text(i, v + 2, f"{v:.0f}%", ha="center")
ax.set_title("Capstone evaluation")
plt.show()

## Step 9: The monitoring dashboard (Day 9)

In [ ]:
usage_df = pd.DataFrame(USAGE_LOG)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].bar(range(1, len(usage_df) + 1), usage_df["tokens"], color="#DD8452")
axes[0].set_title("Tokens per user message")
axes[0].set_xlabel("Message #")
axes[0].set_ylabel("Tokens")

axes[1].barh(list(ALL_TOOL_CALLS.keys()), list(ALL_TOOL_CALLS.values()), color="#4C72B0")
axes[1].set_title("Tool usage this session")
axes[1].set_xlabel("Calls")

plt.tight_layout()
plt.show()

print("Total tokens this session:", int(usage_df["tokens"].sum()))

## Step 10 (optional): a web chat interface

Gradio turns the assistant into a chat web page with a few lines. Run the next cell to build the interface, then the cell after to open it. Stop it with the stop button in Jupyter when you are done.

In this simple UI, ticket requests are auto-approved; in a real deployment the approval would go to a team lead.

In [ ]:
try:
    import gradio as gr

    ui_user = UserContext(employee_id="E101", name="Asha")

    def respond(message, history):
        return chat(message, ui_user, thread_id="gradio-session", return_answer=True)

    demo = gr.ChatInterface(
        fn=respond,
        title="NovaTech AI Assistant",
        description="Ask about NovaTech policies, your leave balance, CloudDesk orders, or raise a ticket.",
        examples=["How many leave days do I have?", "What is the hotel limit in metro cities?",
                  "Status of order ORD-1002?"],
    )
    print("Interface ready. Run the next cell to launch it.")
except ImportError:
    print("Gradio is not installed. Run: pip install gradio")

In [ ]:
demo.launch()

## What you have built

Take a moment to look at the assistant you just built. It:

- searches company documents and cites them (RAG)
- reads trusted user context so data stays private to each employee
- looks up live data and does calculations with tools
- remembers conversations per thread and summarises long ones
- redacts personal data, caps runaway loops, and asks a human before creating tickets
- is evaluated with a test set and monitored for cost

That is a genuine production-style architecture. Real-world versions add a proper database instead of CSV files, a Postgres checkpointer instead of `InMemorySaver`, LangSmith tracing, and a deployment (FastAPI on a cloud service).

## Ideas to extend the capstone

1. Replace `TICKETS` with a SQLite table and add a `ticket_status(ticket_id)` tool.
2. Add a `role` field to `UserContext` and only allow `lookup_order` for the support team.
3. Expand `EVAL_SET` to 20 questions and track accuracy every time you change the prompt.
4. Serve the assistant with FastAPI (Day 9) and connect the Gradio UI to that API.
5. Explore **LangGraph** directly to build custom multi-step workflows and multi-agent systems.

## Series recap

| Day | You learned |
|-----|-------------|
| 1 | Chat models, messages, invoke / stream / batch |
| 2 | Prompt templates, output parsers, structured output |
| 3 | LCEL: pipes, parallel steps, routing, fallbacks |
| 4 | Memory and history management |
| 5 | Loaders, splitters, embeddings, vector stores, retrievers |
| 6 | RAG with sources, conversational RAG, evaluation |
| 7 | Tools and the tool-calling loop |
| 8 | Agents, memory, context, middleware, human approval |
| 9 | Async, caching, cost tracking, callbacks, guardrails, serving |
| 10 | A complete, evaluated, monitored assistant |

Thank you for learning with this series. Keep building, and keep experimenting - the best way to master LangChain is to take these notebooks and break them, change them, and rebuild them for your own ideas.